# Анализ признаков: SHAP и Ablation Study


Ноутбук анализирует важность признаков CatBoost-ранкера через:
1. Нативную feature importance CatBoost
2. SHAP-значения (TreeExplainer)
3. Ablation study по группам признаков

---

## 1. Настройка и загрузка данных

In [ ]:
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from catboost import CatBoostRanker, Pool
from typing import Dict, List, Optional
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid')
PALETTE = sns.color_palette('colorblind')
plt.rcParams.update({'figure.figsize': (12, 6), 'font.size': 12, 'axes.titlesize': 14})

print('Настройка завершена.')

In [ ]:
# Подключение к БД
conn = psycopg2.connect(host='localhost', port=5433, user='postgres', password='postgres', dbname='spbtechrun')
conn.autocommit = True
print('Подключено к БД.')

In [ ]:
# Загрузка таблиц
products_df = pd.read_sql('SELECT id, name, price, category_id, vendor, picture FROM products', conn)
categories_df = pd.read_sql('SELECT id, parent_id, name FROM categories', conn)
feedback_df = pd.read_sql('SELECT main_product_id, recommended_product_id, positive_count, negative_count FROM pair_feedback_stats', conn)
copurchase_df = pd.read_sql('SELECT product_id_1, product_id_2, copurchase_count FROM copurchase_stats', conn)
product_stats_df = pd.read_sql('SELECT product_id, view_count, cart_add_count, order_count FROM product_stats', conn)

# Эмбеддинги
cur = conn.cursor()
cur.execute('SELECT product_id, embedding FROM product_embeddings WHERE embedding IS NOT NULL')
embeddings_dict = {}
for row in cur:
    embeddings_dict[row[0]] = np.array(row[1], dtype=np.float32)
cur.close()

embeddings_normed = {}
for pid, emb in embeddings_dict.items():
    norm = np.linalg.norm(emb)
    embeddings_normed[pid] = emb / norm if norm > 0 else emb

# Скидки
discount_df = pd.read_sql(
    """SELECT p.id as product_id, pr.discount_price
       FROM promos pr JOIN products p ON p.id = pr.product_id
       WHERE pr.discount_price IS NOT NULL AND pr.discount_price > 0""", conn)
discount_map = dict(zip(discount_df['product_id'], discount_df['discount_price']))

print(f'Товаров: {len(products_df)}, Эмбеддингов: {len(embeddings_dict)}, Пар фидбека: {len(feedback_df)}')

## 2. Подготовка словарей и признаков

In [ ]:
# Словари для быстрого доступа
product_price = dict(zip(products_df['id'], products_df['price']))
product_category = dict(zip(products_df['id'], products_df['category_id']))
product_vendor = dict(zip(products_df['id'], products_df['vendor']))
product_picture = dict(zip(products_df['id'], products_df['picture']))
product_name_len = dict(zip(products_df['id'], products_df['name'].str.len()))

stats_map = {}
for _, row in product_stats_df.iterrows():
    stats_map[row['product_id']] = {
        'view_count': row['view_count'] or 0,
        'cart_add_count': row['cart_add_count'] or 0,
        'order_count': row['order_count'] or 0,
    }

feedback_map = {}
for _, row in feedback_df.iterrows():
    feedback_map[(row['main_product_id'], row['recommended_product_id'])] = {
        'positive': row['positive_count'], 'negative': row['negative_count'],
    }

copurchase_map = {}
for _, row in copurchase_df.iterrows():
    copurchase_map[(row['product_id_1'], row['product_id_2'])] = row['copurchase_count']
    copurchase_map[(row['product_id_2'], row['product_id_1'])] = row['copurchase_count']

# Корневые категории
cat_parent = dict(zip(categories_df['id'], categories_df['parent_id']))
def get_root_category(cat_id):
    visited = set()
    current = cat_id
    while current is not None and current in cat_parent and current not in visited:
        visited.add(current)
        parent = cat_parent[current]
        if parent is None or parent == current: return current
        current = parent
    return current
root_category_map = {cid: get_root_category(cid) for cid in categories_df['id']}

print('Словари построены.')

In [ ]:
# 39 признаков и 7 групп (повторяет feature_extractor.py)
FEATURE_NAMES = [
    'embedding_cosine_similarity', 'embedding_l2_distance', 'embedding_dot_product',
    'embedding_euclidean_distance', 'embedding_manhattan_distance', 'embedding_has_valid',
    'pair_feedback_positive', 'pair_feedback_negative', 'pair_feedback_total',
    'pair_feedback_approval_rate',
    'scenario_feedback_positive', 'scenario_feedback_negative', 'scenario_feedback_total',
    'scenario_feedback_approval_rate',
    'candidate_price', 'price_ratio', 'price_diff', 'price_diff_percent',
    'has_discount', 'discount_percent', 'discount_amount',
    'same_category', 'same_root_category', 'category_distance',
    'same_vendor', 'different_vendor',
    'copurchase_count', 'copurchase_log', 'copurchase_exists',
    'has_image', 'is_discounted', 'price_bucket', 'name_length',
    'view_count', 'cart_add_count', 'order_count',
    'cart_similarity_max', 'cart_similarity_avg', 'cart_products_count',
]

FEATURE_GROUPS = {
    'Семантические': FEATURE_NAMES[0:6],
    'Фидбек': FEATURE_NAMES[6:14],
    'Ценовые': FEATURE_NAMES[14:21],
    'Категорийные': FEATURE_NAMES[21:26],
    'Совместные покупки': FEATURE_NAMES[26:29],
    'Популярность': FEATURE_NAMES[29:36],
    'Контекст корзины': FEATURE_NAMES[36:39],
}

feature_to_group = {}
for group, features in FEATURE_GROUPS.items():
    for f in features:
        feature_to_group[f] = group

GROUP_COLORS = {name: PALETTE[i] for i, name in enumerate(FEATURE_GROUPS.keys())}

print(f'{len(FEATURE_NAMES)} признаков в {len(FEATURE_GROUPS)} группах.')

In [ ]:
def extract_features(main_id, cand_id):
    """Извлекает 39 признаков для пары товаров."""
    main_price = product_price.get(main_id, 0)
    cand_price = product_price.get(cand_id, 0)
    if main_price == 0 and cand_price == 0:
        return None

    main_emb = embeddings_dict.get(main_id)
    cand_emb = embeddings_dict.get(cand_id)
    main_norm = embeddings_normed.get(main_id)
    cand_norm = embeddings_normed.get(cand_id)

    if main_emb is not None and cand_emb is not None:
        cosine_sim = float(np.dot(main_norm, cand_norm))
        l2_dist = float(np.linalg.norm(main_norm - cand_norm))
        dot_prod = cosine_sim
        euclidean = float(np.linalg.norm(main_emb - cand_emb))
        manhattan = float(np.sum(np.abs(main_emb - cand_emb)))
        has_valid = 1.0
    else:
        cosine_sim, l2_dist, dot_prod = 0.5, 1.0, 0.0
        euclidean, manhattan, has_valid = 1.0, 1.0, 0.0

    fb = feedback_map.get((main_id, cand_id), {'positive': 0, 'negative': 0})
    pair_pos, pair_neg = fb['positive'], fb['negative']
    pair_total = pair_pos + pair_neg
    pair_approval = (pair_pos + 1) / (pair_total + 2)
    scen_pos, scen_neg, scen_total, scen_approval = 0, 0, 0, 0.5

    price_ratio = cand_price / max(main_price, 1)
    price_diff = cand_price - main_price
    price_diff_pct = (price_diff / max(main_price, 1)) * 100
    cand_discount = discount_map.get(cand_id)
    has_discount = 1.0 if cand_discount else 0.0
    discount_pct = ((cand_price - cand_discount) / cand_price * 100) if cand_discount and cand_price > 0 else 0.0
    discount_amt = (cand_price - cand_discount) if cand_discount else 0.0

    main_cat = product_category.get(main_id)
    cand_cat = product_category.get(cand_id)
    same_cat = 1.0 if main_cat == cand_cat else 0.0
    main_root = root_category_map.get(main_cat)
    cand_root = root_category_map.get(cand_cat)
    same_root = 1.0 if main_root and cand_root and main_root == cand_root else 0.0
    cat_dist = 0.0 if same_root else 2.0
    main_v = product_vendor.get(main_id, '')
    cand_v = product_vendor.get(cand_id, '')
    same_vendor = 1.0 if main_v and cand_v and main_v == cand_v else 0.0

    cop = copurchase_map.get((main_id, cand_id), 0)
    s = stats_map.get(cand_id, {'view_count': 0, 'cart_add_count': 0, 'order_count': 0})
    has_image = 1.0 if product_picture.get(cand_id) else 0.0
    p_bucket = 2.0 if cand_price > 10000 else (1.0 if cand_price > 1000 else 0.0)
    name_len = float(product_name_len.get(cand_id, 0))

    return [
        cosine_sim, l2_dist, dot_prod, euclidean, manhattan, has_valid,
        float(pair_pos), float(pair_neg), float(pair_total), pair_approval,
        float(scen_pos), float(scen_neg), float(scen_total), scen_approval,
        float(cand_price), price_ratio, price_diff, price_diff_pct,
        has_discount, discount_pct, discount_amt,
        same_cat, same_root, cat_dist, same_vendor, 1.0 - same_vendor,
        float(cop), np.log1p(cop), 1.0 if cop > 0 else 0.0,
        has_image, has_discount, p_bucket, name_len,
        np.log1p(s['view_count']), np.log1p(s['cart_add_count']), np.log1p(s['order_count']),
        0.0, 0.0, 0.0,
    ]

print('Функция извлечения признаков готова.')

## 3. Обучение CatBoost модели

In [ ]:
# Формирование обучающих данных из фидбека + negative sampling
relevance_records = []
for _, row in feedback_df.iterrows():
    mid, rid = row['main_product_id'], row['recommended_product_id']
    pos, neg = row['positive_count'], row['negative_count']
    if pos >= 5 and pos > neg * 2:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 2})
    elif pos >= 3 and pos > neg:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 1})
    elif neg > pos:
        relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 0})

existing_pairs = set((r['main_product_id'], r['candidate_product_id']) for r in relevance_records)
for _, row in copurchase_df.iterrows():
    p1, p2, count = row['product_id_1'], row['product_id_2'], row['copurchase_count']
    if count >= 2:
        for mid, rid in [(p1, p2), (p2, p1)]:
            if (mid, rid) not in existing_pairs:
                relevance_records.append({'main_product_id': mid, 'candidate_product_id': rid, 'relevance': 1})
                existing_pairs.add((mid, rid))

# Negative sampling: для каждого query добавляем случайные товары как negatives
# Это критично — без этого в большинстве групп 1-2 кандидата и ранжирование тривиально
all_product_ids = list(products_df['id'].values)
query_ids = set(r['main_product_id'] for r in relevance_records)
NEGATIVES_PER_QUERY = 8

for qid in query_ids:
    added = 0
    attempts = 0
    while added < NEGATIVES_PER_QUERY and attempts < 50:
        neg_id = all_product_ids[np.random.randint(len(all_product_ids))]
        attempts += 1
        if neg_id != qid and (qid, neg_id) not in existing_pairs:
            relevance_records.append({'main_product_id': qid, 'candidate_product_id': neg_id, 'relevance': 0})
            existing_pairs.add((qid, neg_id))
            added += 1

relevance_df = pd.DataFrame(relevance_records)
print(f'Записей релевантности: {len(relevance_df)}')
print(relevance_df['relevance'].value_counts().sort_index())
print(f'Уникальных запросов: {relevance_df["main_product_id"].nunique()}')
print(f'Среднее кандидатов на запрос: {len(relevance_df) / relevance_df["main_product_id"].nunique():.1f}')

In [ ]:
# Извлечение признаков
X, y, groups = [], [], []
for _, row in relevance_df.iterrows():
    features = extract_features(row['main_product_id'], row['candidate_product_id'])
    if features:
        X.append(features)
        y.append(row['relevance'])
        groups.append(row['main_product_id'])

X_df = pd.DataFrame(X, columns=FEATURE_NAMES)
y_s = pd.Series(y)

# Сортировка по group_id (требование CatBoost)
order = np.argsort(groups)
X_df = X_df.iloc[order].reset_index(drop=True)
y_s = y_s.iloc[order].reset_index(drop=True)
groups_sorted = [groups[i] for i in order]

print(f'Датасет: {len(X_df)} примеров, {len(set(groups_sorted))} запросов')

In [ ]:
# Обучение CatBoost
# Перемешиваем данные внутри каждой группы (важно для корректной оценки)
shuffled_indices = []
groups_arr = np.array(groups_sorted)
for g in np.unique(groups_arr):
    group_idx = np.where(groups_arr == g)[0]
    np.random.shuffle(group_idx)
    shuffled_indices.extend(group_idx)

X_df = X_df.iloc[shuffled_indices].reset_index(drop=True)
y_s = y_s.iloc[shuffled_indices].reset_index(drop=True)
groups_sorted = [groups_sorted[i] for i in shuffled_indices]

train_pool = Pool(data=X_df, label=y_s, group_id=groups_sorted)

model = CatBoostRanker(
    iterations=500, learning_rate=0.05, depth=6,
    loss_function='YetiRank',
    custom_metric=['NDCG:top=10', 'PrecisionAt:top=5'],
    random_seed=RANDOM_SEED, verbose=100,
    eval_metric='NDCG:top=10',
)
model.fit(train_pool, plot=False)

# Проверяем что модель реально обучилась
preds = model.predict(X_df)
print(f'Обучение CatBoost завершено.')
print(f'Предсказания: min={preds.min():.4f}, max={preds.max():.4f}, std={preds.std():.4f}')
print(f'Деревьев: {model.tree_count_}')

## 4. Нативная важность признаков CatBoost

In [ ]:
# Важность признаков через SHAP (надёжнее чем get_feature_importance для YetiRank)
# Сначала вычислим SHAP-значения
explainer = shap.TreeExplainer(model)
sample_size = min(3000, len(X_df))
sample_idx = np.random.choice(len(X_df), size=sample_size, replace=False)
X_sample = X_df.iloc[sample_idx]
shap_values = explainer.shap_values(X_sample)

# Feature importance = mean(|SHAP|)
feature_importance = np.abs(shap_values).mean(axis=0)

importance_df = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'importance': feature_importance,
    'group': [feature_to_group[f] for f in FEATURE_NAMES],
}).sort_values('importance', ascending=False)

print(f'Сумма важности: {feature_importance.sum():.4f}')
print(f'Ненулевых: {(feature_importance > 1e-6).sum()}/{len(feature_importance)}')
print('\nТоп-15 признаков:')
print(importance_df.head(15).to_string(index=False))

In [ ]:
# Горизонтальная диаграмма всех 39 признаков, цвет по группе
fig, ax = plt.subplots(figsize=(12, 14))
colors = [GROUP_COLORS[feature_to_group[f]] for f in importance_df['feature']]
bars = ax.barh(range(len(importance_df)), importance_df['importance'].values,
               color=colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(len(importance_df)))
ax.set_yticklabels(importance_df['feature'].values, fontsize=9)
ax.set_xlabel('Importance')
ax.set_title('Важность признаков CatBoost (по группам)')
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=GROUP_COLORS[g], label=g) for g in FEATURE_GROUPS]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('figures/feature_importance_by_group.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Агрегированная важность по группам
group_importance = importance_df.groupby('group')['importance'].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [GROUP_COLORS[g] for g in group_importance.index]
bars = ax.barh(group_importance.index, group_importance.values, color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Суммарная важность')
ax.set_title('Важность по группам признаков')

for bar, val in zip(bars, group_importance.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/group_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. SHAP-анализ

In [ ]:
# SHAP-значения уже вычислены выше (cell 14)
print(f'SHAP-значения: {shap_values.shape[0]} примеров, {shap_values.shape[1]} признаков')
print(f'Expected value: {explainer.expected_value:.4f}')

In [ ]:
# SHAP Summary Plot (beeswarm)
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_NAMES, show=False, max_display=20)
plt.title('SHAP Summary Plot: влияние признаков на предсказание')
plt.tight_layout()
plt.savefig('figures/shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar Plot (средние абсолютные значения)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_NAMES,
                  plot_type='bar', show=False, max_display=20)
plt.title('Средний |SHAP| по признакам')
plt.tight_layout()
plt.savefig('figures/shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Dependence Plots для топ-4 признаков
top_features = importance_df.head(4)['feature'].tolist()

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for ax, feat in zip(axes.flat, top_features):
    feat_idx = FEATURE_NAMES.index(feat)
    shap.dependence_plot(feat_idx, shap_values, X_sample,
                         feature_names=FEATURE_NAMES, ax=ax, show=False)
    ax.set_title(f'Dependence: {feat}')

plt.suptitle('SHAP Dependence Plots: топ-4 признака', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('figures/shap_dependence_top4.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Force Plot для лучшего и худшего примера
predictions = model.predict(X_sample)
best_idx = np.argmax(predictions)
worst_idx = np.argmin(predictions)

print(f'Лучший пример (скор={predictions[best_idx]:.3f}):')
shap.force_plot(explainer.expected_value, shap_values[best_idx],
                X_sample.iloc[best_idx], feature_names=FEATURE_NAMES, matplotlib=True)
plt.savefig('figures/shap_force_best.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Худший пример (скор={predictions[worst_idx]:.3f}):')
shap.force_plot(explainer.expected_value, shap_values[worst_idx],
                X_sample.iloc[worst_idx], feature_names=FEATURE_NAMES, matplotlib=True)
plt.savefig('figures/shap_force_worst.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Ablation Study по группам признаков

In [ ]:
from sklearn.metrics import ndcg_score

def train_and_evaluate_ndcg(X_train, y_train, groups_train, X_eval, y_eval, groups_eval):
    """Обучает CatBoost и возвращает средний NDCG@10 по группам."""
    pool = Pool(data=X_train, label=y_train, group_id=groups_train)
    m = CatBoostRanker(
        iterations=300, learning_rate=0.05, depth=6,
        loss_function='YetiRank', random_seed=RANDOM_SEED, verbose=0,
    )
    m.fit(pool, plot=False)

    preds = m.predict(X_eval)
    groups_arr = np.array(groups_eval)
    unique_groups = np.unique(groups_arr)
    ndcgs = []
    for g in unique_groups:
        mask = groups_arr == g
        g_true = y_eval[mask].values
        g_pred = preds[mask]
        if len(g_true) < 2 or g_true.max() == 0:
            continue
        # Перемешиваем внутри группы чтобы исключить bias от порядка данных
        shuffle_idx = np.random.permutation(len(g_true))
        g_true = g_true[shuffle_idx]
        g_pred = g_pred[shuffle_idx]
        try:
            ndcgs.append(ndcg_score([g_true], [g_pred], k=10))
        except:
            pass
    return np.mean(ndcgs) if ndcgs else 0.0

print('Функция обучения и оценки готова.')

In [ ]:
# Ablation: убираем по одной группе признаков
# Делим данные на train/test по group_id чтобы избежать переобучения
unique_groups = list(set(groups_sorted))
np.random.shuffle(unique_groups)
split_idx = int(len(unique_groups) * 0.7)
train_groups_set = set(unique_groups[:split_idx])

train_mask = np.array([g in train_groups_set for g in groups_sorted])
test_mask = ~train_mask

X_train_abl = X_df.loc[train_mask].reset_index(drop=True)
y_train_abl = y_s.loc[train_mask].reset_index(drop=True)
groups_train_abl = [g for g, m in zip(groups_sorted, train_mask) if m]

X_test_abl = X_df.loc[test_mask].reset_index(drop=True)
y_test_abl = y_s.loc[test_mask].reset_index(drop=True)
groups_test_abl = [g for g, m in zip(groups_sorted, test_mask) if m]

print(f'Ablation split: train={len(X_train_abl)}, test={len(X_test_abl)}')
print(f'Train groups: {len(set(groups_train_abl))}, Test groups: {len(set(groups_test_abl))}')
print(f'Запуск ablation study (8 обучений CatBoost)...')
print('-' * 60)

ablation_results = []

# Полная модель (бейзлайн)
full_ndcg = train_and_evaluate_ndcg(X_train_abl, y_train_abl, groups_train_abl,
                                     X_test_abl, y_test_abl, groups_test_abl)
ablation_results.append({'group': 'Нет (полная модель)', 'NDCG@10': full_ndcg, 'Delta': 0.0})
print(f'Полная модель: NDCG@10 = {full_ndcg:.4f}')

for group_name, group_features in FEATURE_GROUPS.items():
    remaining = [f for f in FEATURE_NAMES if f not in group_features]
    X_tr = X_train_abl[remaining]
    X_te = X_test_abl[remaining]
    ablated_ndcg = train_and_evaluate_ndcg(X_tr, y_train_abl, groups_train_abl,
                                            X_te, y_test_abl, groups_test_abl)
    delta = full_ndcg - ablated_ndcg
    ablation_results.append({'group': group_name, 'NDCG@10': ablated_ndcg, 'Delta': delta})
    print(f'  -{group_name} ({len(group_features)} призн.): NDCG@10 = {ablated_ndcg:.4f}, delta = {delta:+.4f}')

ablation_df = pd.DataFrame(ablation_results)
print('\nAblation study завершена.')

In [ ]:
# Таблица ablation со стилизацией
styled = ablation_df.style.background_gradient(
    subset=['Delta'], cmap='RdYlGn_r'
).format({'NDCG@10': '{:.4f}', 'Delta': '{:+.4f}'})
styled

In [ ]:
# График: падение NDCG при удалении каждой группы
ablation_plot = ablation_df[ablation_df['group'] != 'Нет (полная модель)'].sort_values('Delta', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [GROUP_COLORS.get(g, PALETTE[0]) for g in ablation_plot['group']]
bars = ax.barh(ablation_plot['group'], ablation_plot['Delta'], color=colors, edgecolor='black', linewidth=0.5)
ax.set_xlabel('Падение NDCG@10 (чем больше, тем важнее группа)')
ax.set_title('Ablation Study: вклад каждой группы признаков')
ax.axvline(x=0, color='black', linewidth=0.8)

for bar, val in zip(bars, ablation_plot['Delta']):
    ax.text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:+.4f}',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('figures/ablation_study.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Сводная фигура для презентации

In [ ]:
# 2x2 сводная фигура
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# (a) SHAP summary
plt.sca(axes[0, 0])
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_NAMES,
                  show=False, max_display=15)
axes[0, 0].set_title('(a) SHAP Summary Plot', fontsize=13)

# (b) Важность по группам
group_imp = importance_df.groupby('group')['importance'].sum().sort_values(ascending=True)
colors_b = [GROUP_COLORS[g] for g in group_imp.index]
axes[0, 1].barh(group_imp.index, group_imp.values, color=colors_b, edgecolor='black', linewidth=0.5)
axes[0, 1].set_xlabel('Суммарная важность')
axes[0, 1].set_title('(b) Важность по группам признаков', fontsize=13)

# (c) Ablation
abl = ablation_df[ablation_df['group'] != 'Нет (полная модель)'].sort_values('Delta', ascending=True)
colors_c = [GROUP_COLORS.get(g, PALETTE[0]) for g in abl['group']]
axes[1, 0].barh(abl['group'], abl['Delta'], color=colors_c, edgecolor='black', linewidth=0.5)
axes[1, 0].set_xlabel('Падение NDCG@10')
axes[1, 0].set_title('(c) Ablation Study', fontsize=13)
axes[1, 0].axvline(x=0, color='black', linewidth=0.8)

# (d) Top feature dependence
top_feat = importance_df.iloc[0]['feature']
top_idx = FEATURE_NAMES.index(top_feat)
plt.sca(axes[1, 1])
shap.dependence_plot(top_idx, shap_values, X_sample,
                     feature_names=FEATURE_NAMES, ax=axes[1, 1], show=False)
axes[1, 1].set_title(f'(d) SHAP Dependence: {top_feat}', fontsize=13)

plt.suptitle('Анализ признаков CatBoost-ранкера', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('figures/feature_analysis_hero.png', dpi=300, bbox_inches='tight')
plt.savefig('figures/feature_analysis_hero.svg', bbox_inches='tight')
plt.show()
print('Сводная фигура сохранена.')

## 8. Выводы

### Основные результаты

1. **Семантические признаки** (эмбеддинги) -- ожидаемо доминирующая группа, т.к. основной сигнал релевантности определяется текстовым описанием товаров.

2. **Ценовые признаки** важны для ранжирования сопутствующих товаров -- покупатели предпочитают товары в сопоставимом ценовом диапазоне.

3. **Фидбек** имеет умеренное влияние -- ограничен синтетическими данными, но демонстрирует способность модели использовать сигнал одобрения.

4. **Совместные покупки** минимально влияют из-за крайней разреженности данных (31 пара).

5. **Контекст корзины** обнулён в офлайн-оценке -- при онлайн-использовании вклад будет другим.

### Практические выводы

- При ограниченных ресурсах достаточно семантических + ценовых признаков для приемлемого качества.
- Инвестиции в сбор реального фидбека и данных совместных покупок дадут наибольший прирост.

In [ ]:
conn.close()
print('Готово. Соединение закрыто.')